# 04 — Expanded selected-dataset diagnostic suite
Run All: five-class Kinetics-600 (>2,000 unique videos before splitting) → temporal audit → tiny training-only learning check → model screen and one-factor comparisons → two-seed confirmation → freeze → test metrics and videos. An eight-hour cooperative compute deadline starts after data setup; incomplete studies cannot open test.

Saved outputs below are historical, not results of this plan. This exploratory diagnostic is separate from the unchanged controlled AAD protocol and original reference notebook. Do not tune from test results.

In [2]:
from pathlib import Path
import subprocess
import sys
import importlib
import os
os.environ.setdefault('CUBLAS_WORKSPACE_CONFIG', ':4096:8')

REPO_ROOT = Path('/content/phd_research')
REPO_ROOT.parent.mkdir(parents=True, exist_ok=True)
if not (REPO_ROOT / '.git').exists():
    if REPO_ROOT.exists() and any(REPO_ROOT.iterdir()):
        raise RuntimeError(f'{REPO_ROOT} contains files but is not a Git checkout. Use an empty checkout path; existing files were preserved.')
    subprocess.run(['git', 'clone', 'https://github.com/sanelehlabisa/phd_research.git', str(REPO_ROOT)], check=True)
else:
    update = subprocess.run(['git', 'pull', '--ff-only', 'origin', 'master'], cwd=REPO_ROOT, capture_output=True, text=True)
    print(update.stdout, flush=True)
    if update.returncode:
        raise RuntimeError('Cannot safely update the Colab checkout. Save/commit conflicting local edits, resolve the Git error, then rerun setup. No local edits were discarded.\n' + update.stderr)
IMPLEMENTATION_ROOT = REPO_ROOT / 'papers/001-journal-abnormal-activity-recognition/implementation'
required_helpers = ('notebook_data.py', 'notebook_display.py', 'notebook_workflows.py', 'kinetics600_subset.py', 'notebook_models.py', 'notebook_suite.py', 'notebook_diagnostics.py')
missing_helpers = [name for name in required_helpers if not (IMPLEMENTATION_ROOT / 'src' / name).is_file()]
revision = subprocess.run(['git', 'rev-parse', '--short', 'HEAD'], cwd=REPO_ROOT, capture_output=True, text=True, check=True).stdout.strip()
print(f'Code checkout: {REPO_ROOT} @ {revision}', flush=True)
print(f'Notebook Python: {sys.executable}', flush=True)
try:
    gpu = subprocess.run(['nvidia-smi', '--query-gpu=name', '--format=csv,noheader'], capture_output=True, text=True, timeout=15)
    print('Attached GPU:', gpu.stdout.strip() or gpu.stderr.strip(), flush=True)
except (OSError, subprocess.TimeoutExpired) as error:
    print('GPU check:', error, flush=True)
if missing_helpers:
    raise RuntimeError(f'Colab checkout is missing repository files: {missing_helpers}. These are NOT pip packages. Commit and push the complete notebook/source update to master, then rerun this setup cell. A kernel restart alone cannot download unpublished files.')
bootstrap = subprocess.run([sys.executable, str(IMPLEMENTATION_ROOT / 'src/colab_bootstrap.py'), str(IMPLEMENTATION_ROOT / 'requirements.txt')])
if bootstrap.returncode == 75:
    raise SystemExit('Pinned packages changed. Restart this notebook kernel once, then rerun from the top. Keep the same A100 runtime if it is still attached; reselect it only if VS Code asks.')
bootstrap.check_returncode()
sys.path[:] = [str(IMPLEMENTATION_ROOT)] + [entry for entry in sys.path if entry != str(IMPLEMENTATION_ROOT)]
for name in [name for name in sys.modules if name == 'src' or name.startswith('src.')]:
    del sys.modules[name]
importlib.invalidate_caches()
from src.notebook_utils import validate_runtime
print(validate_runtime(require_cuda=True))
from src.notebook_suite import prepare_experiment_data
prepared = prepare_experiment_data(IMPLEMENTATION_ROOT)


Already up to date.

Code checkout: /content/phd_research @ 16dcf3c
Notebook Python: /usr/bin/python3
Attached GPU: NVIDIA A100-SXM4-80GB
{'python': '3.13.15', 'platform': 'Linux-6.6.122+-x86_64-with-glibc2.39', 'pytorch': '2.9.1+cu128', 'torchvision': '0.24.1+cu128', 'cuda_available': True, 'cuda_version': '12.8', 'cudnn_version': 91002, 'cuda_device': 'NVIDIA A100-SXM4-80GB'}
Dataset: kinetics600-subset | https://s3.amazonaws.com/kinetics/600/train
Cached archive: headbutting.tar.gz
Cached class: headbutting (535 clips)
Cached archive: slapping.tar.gz
Cached class: slapping (872 clips)
Cached archive: punching person (boxing).tar.gz
Cached class: punching person (boxing) (496 clips)
Cached archive: hugging (not baby).tar.gz
Cached class: hugging (not baby) (521 clips)
Cached archive: shaking hands.tar.gz
Cached class: shaking hands (619 clips)
  headbutting: 535 unique clips


ValueError: Duplicate video content: /content/phd_research/papers/001-journal-abnormal-activity-recognition/implementation/runs/datasets/kinetics600-five-activities/files/kinetics-600-train/slapping/U6TLfmF_pxE_000040_000050.mp4; copies cannot count as new data

## Declared candidates, factors and budgets
Screen 11 custom models and three scratch 3D CNNs for 24 epochs at 64×64, 32 frames/8 FPS. Against the validation-selected custom reference, change one factor at a time: 64/96/128 pixels, 4/8/16 FPS coverage, dropout 0/0.1/0.5 and weight decay 0/0.0001/0.001. Try the native published topology separately without shrinking it. The displayed plan includes run counts; the eight-hour cap may stop before all runs finish. Configure this suite with SUITE_* settings, not the retained legacy quick-grid SCREEN_* lists.

In [ ]:
from src.notebook_suite import show_experiment_plan, run_screen, train_winner, final_evaluate
screen_dir = final_data = None  # Clear any previous run before starting.
plan = show_experiment_plan(prepared)


In [ ]:
screen_dir = run_screen(prepared)


## Longer confirmation and validation-only freeze
Train the selected custom configuration and all three CNNs from scratch for 64 epochs, seeds 42/2026, at max(96, selected spatial size). Use identical splits, sampling and budgets; report mean/std validation metrics. Freeze every compatible checkpoint before opening test. No test evaluation after an incomplete suite.

In [ ]:
final_data = train_winner(prepared, screen_dir)


## Final test metrics, then playable test predictions
Evaluate frozen confirmations without reranking from test. Show up to five test videos/probabilities from the validation winner's predeclared seed 42. Metrics and reports are saved before display; rerunning this cell reuses saved results. These diagnostics are not independent surveillance-paper evidence.

In [ ]:
final_report = final_evaluate(final_data, screen_dir)
